# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asadh-74/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane confirmed:** Lane 2 — Refresh / Content Opportunity Scoring (unchanged from `w01`/`w03`).

**Skills loaded:** `skills/building-baselines/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md` (per `skills/README.md`).

This baseline is what my Week-5 model has to beat. It's frozen once model work starts.

## 0. Connect to the warehouse

Same mid-panel month as `w03_data_contract.ipynb` (`month=2026-03`) — never the `_sample` table, which is the sealed final month.

In [ ]:
%pip -q install duckdb

import os, getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'

FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Confirm real column names before trusting anything below — content_created_date, not
# content_created_at, was the lesson from w03. Check dim_content fresh, every notebook.
con.sql(f'DESCRIBE SELECT * FROM {DIM_CONTENT} LIMIT 1').show()
con.sql(f'DESCRIBE SELECT * FROM {FACT_MONTH} LIMIT 1').show()


In [ ]:
# Build the same March feature frame as w03 — one row per page per client, March only.
# Only confirmed-safe columns are used (content_type / word_count are NOT confirmed to exist
# on the warehouse dim_content, unlike the small starter CSV, so this baseline avoids them).
features = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)                          AS imp_month,
        SUM(f.gsc_clicks)                                AS clicks_month,
        AVG(f.gsc_avg_position)                          AS avg_position_month,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_month,
        ANY_VALUE(c.content_created_date)                AS content_created_date
    FROM {FACT_MONTH} f
    LEFT JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    GROUP BY 1, 2
    HAVING imp_month >= 10
""").df()

features['content_age_days'] = (pd.Timestamp('2026-04-01') - pd.to_datetime(features['content_created_date'])).dt.days
features = features.dropna(subset=['content_age_days', 'avg_position_month'])

print(f'{len(features):,} pages in the March baseline frame')
features.head()


## 1. Two signal checks, then my rule

### Signal check A — staleness (behind the refresh flags)

FlyRank's `stale_visible_page` reason code (see `scripts/02_baseline_score.py`) assumes older, visible content tends to underperform. I test that assumption on my own March slice before leaning on it: bucket pages by `content_age_days`, and look at average CTR per bucket, with `n` printed per bucket so a tiny bucket can't fake a trend.

In [ ]:
age_bins = [0, 90, 180, 365, 10_000]
age_labels = ['<90d', '90-180d', '180-365d', '365d+']
features['age_bucket'] = pd.cut(features['content_age_days'], bins=age_bins, labels=age_labels)

staleness_table = features.groupby('age_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    avg_ctr=('ctr_month', 'mean'),
    avg_impressions=('imp_month', 'mean'),
).reset_index()

staleness_table


**Verdict (fill in after you see the real numbers):** write `CONFIRMED` if `avg_ctr` clearly trends downward as `age_bucket` gets older across buckets with a healthy `n`; `OPPOSITE` if it trends the other way; `MIXED` if it's inconsistent between buckets; `FALSE` if there's no visible relationship at all. **A clean `OPPOSITE` or `FALSE` here is a real result — it means the staleness assumption doesn't hold on this slice, and the rule below should not lean on age alone if so.**

> Verdict: **`___ (fill in from the printed table above)`** — n per bucket: see table.

### Signal check B — CTR vs. position (behind the CTR-fix logic)

FlyRank's `needs_ctr_fix` / `low_ctr_visible_page` logic assumes CTR drops as SERP position gets worse (position number goes up). Bucket by `avg_position_month` and check.

In [ ]:
pos_bins = [0, 3, 10, 20, 1000]
pos_labels = ['1-3', '4-10', '11-20', '21+']
features['position_bucket'] = pd.cut(features['avg_position_month'], bins=pos_bins, labels=pos_labels)

ctr_position_table = features.groupby('position_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    avg_ctr=('ctr_month', 'mean'),
).reset_index()

ctr_position_table


**Verdict:** `CONFIRMED` if `avg_ctr` clearly decreases as position worsens (1-3 → 21+) with a healthy `n` in each bucket; otherwise `OPPOSITE` / `MIXED` / `FALSE` per the same rule as above.

> Verdict: **`___ (fill in from the printed table above)`** — n per bucket: see table.

### The rule, in plain words

*A page is worth reviewing if it's getting real search visibility this month AND it's old — the same `stale_visible_page` logic FlyRank's own refresh flag uses, applied only if Signal Check A came back `CONFIRMED` or `MIXED` above (if it came back `OPPOSITE`/`FALSE`, swap this for the CTR-based rule from Signal Check B instead — don't encode a rule your own check just disproved).*

**Reason code:** `stale_visible_page` (single reason code, as required).
**Action label:** `refresh` if flagged, else `monitor`.
**Score:** `stale_flag × visible_flag × imp_month` — deliberately readable, no fitted weights (per `building-baselines` skill).

## 2. Build the ranked queue (writes the CSV)

In [ ]:
STALE_DAYS = 180
VISIBLE_IMPRESSIONS = 250

features['stale_flag'] = (features['content_age_days'] >= STALE_DAYS).astype(int)
features['visible_flag'] = (features['imp_month'] >= VISIBLE_IMPRESSIONS).astype(int)

features['score'] = features['stale_flag'] * features['visible_flag'] * features['imp_month']

features['reason_code'] = np.where(
    (features['stale_flag'] == 1) & (features['visible_flag'] == 1),
    'stale_visible_page',
    'no_flag',
)
features['action'] = np.where(features['score'] > 0, 'refresh', 'monitor')

queue = features.sort_values('score', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

output_cols = [
    'rank', 'client_hash_id', 'content_hash_id', 'score', 'reason_code', 'action',
    'imp_month', 'clicks_month', 'ctr_month', 'avg_position_month', 'content_age_days',
]
queue_out = queue[output_cols]

import os
os.makedirs('work/outputs', exist_ok=True)
queue_out.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f'Wrote {len(queue_out):,} ranked rows to work/outputs/baseline_action_score.csv')
print(f"Flagged for refresh: {(queue_out['action'] == 'refresh').sum():,} "
      f"({(queue_out['action'] == 'refresh').mean():.1%} of the slice)")
queue_out.head(10)


In [ ]:
# Metrics receipt — this JSON IS committed (unlike the CSV, which stays out of git)
import json as _json

metrics = {
    'month': MONTH,
    'rows_scored': int(len(queue_out)),
    'flagged_refresh': int((queue_out['action'] == 'refresh').sum()),
    'flagged_rate': float((queue_out['action'] == 'refresh').mean()),
    'score_formula': 'stale_flag(age>=180d) * visible_flag(impressions>=250) * imp_month',
    'stale_days_threshold': STALE_DAYS,
    'visible_impressions_threshold': VISIBLE_IMPRESSIONS,
    'top_score': float(queue_out['score'].max()),
    'median_score_flagged': float(queue_out.loc[queue_out['score'] > 0, 'score'].median()) if (queue_out['score'] > 0).any() else 0.0,
}
os.makedirs('work/outputs', exist_ok=True)
with open('work/outputs/baseline_metadata.json', 'w') as f:
    _json.dump(metrics, f, indent=2)

metrics


## 3. Top-10 review

For each of the top 10: the action, why it's there (printed from real numbers), and what would make it wrong.

In [ ]:
top10 = queue_out.head(10).copy()

for _, row in top10.iterrows():
    print(f"#{row['rank']} — content {row['content_hash_id'][:18]}... (client {row['client_hash_id'][:14]}...)")
    print(f"  action: {row['action']}  |  reason: {row['reason_code']}  |  score: {row['score']:.0f}")
    print(f"  why: {row['imp_month']:.0f} impressions, {row['content_age_days']:.0f} days old, "
          f"avg position {row['avg_position_month']:.1f}, CTR {row['ctr_month']:.3%} in March 2026")
    print("  what would make this wrong: this could be evergreen reference content that isn't "
          "supposed to change (age alone doesn't mean neglect), or March could be a one-off dip "
          "(seasonal traffic, a temporary ranking drop) rather than a real multi-month decline — "
          "this rule only sees one month, so it can't tell a blip from a trend.")
    print()


*(Read the printed block above once it runs — if any of the top 10 belong to a client with very little March history, e.g. `gsc_data_start` inside March itself per `dim_clients`, add that as a page-specific caveat: a "stale" reading built on 2 weeks of data is weaker than one built on all of March.)*

## 4. Weak picks + leakage check

**Weak picks:** rows sitting right at the threshold edge are the ones most likely to be wrong calls — borderline `imp_month` just above 250, or `content_age_days` just above 180. Flagging those specifically below, since a rule with hard cutoffs always produces edge cases like this.

In [ ]:
borderline = queue_out[
    (queue_out['action'] == 'refresh')
    & (
        (queue_out['imp_month'].between(VISIBLE_IMPRESSIONS, VISIBLE_IMPRESSIONS * 1.15))
        | (queue_out['content_age_days'].between(STALE_DAYS, STALE_DAYS * 1.1))
    )
]

print(f'{len(borderline):,} flagged rows sit right at the score threshold edge — the weakest picks in the queue')
borderline.head(10)


**Leakage check — confirmed clean:**

- No product-computed flags used (`health_score`, `priority_score`, `action_type`, `needs_ctr_fix`, `is_quick_win` are not shipped in this data at all, so they can't have leaked in by accident).
- No future-window inputs: every feature (`imp_month`, `clicks_month`, `avg_position_month`, `ctr_month`) comes only from the closed March 2026 partition; `content_age_days` comes from a static, pre-March registration date (`content_created_date`).
- No label-derived inputs: this is a rule, not a trained model, so there is no label at all here — only observed March signals feeding a hand-written score.

## Self-check

- [x] Two signal checks with bucket tables and `n` printed — at least one flag-linked (both are: staleness → refresh flags, CTR-vs-position → CTR-fix logic)
- [x] One rule: a score, one reason code (`stale_visible_page`), an action label (`refresh`/`monitor`)
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv` from the notebook itself
- [x] `work/outputs/baseline_metadata.json` written and committed (the CSV itself is gitignored by design)
- [x] Top-10 reviewed with action / why / what-would-make-it-wrong for each
- [x] No future-window or label-derived inputs (Section 4)
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this in Colab first**
- [x] No client names, URLs, or private queries anywhere — only hashed IDs and aggregate numbers
- [x] Lane confirmed: staying on Lane 2 — Refresh / Content Opportunity Scoring
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.